In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.io import fits
from astropy.table import Table
import os
import requests
from tqdm import tqdm


fits_file_path = './astraAllStarASPCAP-0.6.0.fits.gz'


In [2]:
with fits.open(fits_file_path) as hdul:
    table = Table(hdul[2].data)

# Konwersja do pandas
names = [name for name in table.colnames if len(table[name].shape) <= 1]
df = table[names].to_pandas()

# Wersja tylko skalarna
scalar_cols = [col for col in df.columns if not hasattr(df[col].iloc[0], '__len__')]
df_scalar = df[scalar_cols]

In [3]:
clean_stars = df_scalar[df_scalar['snr'] > 100]
print(len(clean_stars))

747904


In [4]:
s = clean_stars["sdss_id"]
s.reset_index(drop=True)

0         85995134
1         55558184
2         69701733
3         54393951
4         54393951
            ...   
747899    66083125
747900    69185616
747901    86425128
747902    62976634
747903    71166250
Name: sdss_id, Length: 747904, dtype: int64

In [5]:

def format_mwm_star(sdss_id):
    base_url = "https://data.sdss.org/sas/dr19/spectro/astra/0.6.0/spectra/star"
    sdss_id_str = str(sdss_id)
    
    path_group = f"{sdss_id_str[-4:-2]}/{sdss_id_str[-2:]}"
    file_url = f"{base_url}/{path_group}/mwmStar-0.6.0-{sdss_id}.fits"
    
    return file_url

In [6]:
with open("DownloadAddresses.txt","w") as file:
    for index, value in s.items():
        file.write(format_mwm_star(value))
        file.write("\n")
